In [116]:
from FormUtils import pyForm, capture_physics_expr

In [ ]:
%%pyForm Renormalization

* Process: Renormalization

#-
* Above suppresses extra output
Off Statistics;
Off FinalStats;
#preout Off
#include SquareAmplitude.h
Vectors k1,k2,q,v1;
Symbols e, Mmuon, Melec;
Symbols Den0, Den1;


Local MLO = (e^2) * (UB(i1, p3, Melec ) * g(i1, i2, mu1) * U(i2, p1, Melec))
            * phprop(mu1, mu2, q)
            * (UB(i3, p4, Mmuon) * g(i3, i4, mu2) * U(i4, p2, Mmuon));

Local MNLO = -1* (e^4) 
             * (UB(i1, p3, Melec) * g(i1, i2, mu1) * U(i2, p1, Melec)) 
             * phprop(mu1, mu3, q) 
             * g(i11, i12, mu3) * fprop(i12, i13, k1, Melec) 
             * g(i13, i14, mu4) * fprop(i14, i11, k2, Melec)
             * phprop(mu4, mu2, q)
             * (UB(i7, p4, Mmuon) * g(i7, i8, mu2) * U(i8, p2, Mmuon));

Local MTotal = MLO+MNLO;
#call squareamplitude(MLO, MsqLO)
#call squareamplitude(MNLO, MsqNLO)
#call squareamplitude(MTotal, MsqTotal)
.sort
Multiply 1/4;
.sort
Drop drop MsqNLO, MsqTotal, MLO, MNLO, MTotal ;
Local MInt = MsqTotal - MsqLO - MsqNLO;
.sort

* --- KINEMATIC DEFINITIONS ---
* q  = p1 - p3           : Momentum transfer between electron and muon lines
* t  = q.q               : Mandelstam variable t (photon momentum squared)
* k  = loop momentum     : Integration variable for the fermion loop
* kf1 = k                : Momentum of the first fermion in the bubble
* kf2 = k - q            : Momentum of the second fermion in the bubble

* --- Passarino Veltman ---
id prop(- Melec^2 + k1.k1 ) = 1/Den0;
id prop( - Melec^2 + k2.k2) = 1/Den1;
.sort
id k2 = k1 - q;
.sort:linearize;

* Replace k1.k1
id k1.k1 = Den0 + Melec^2;

* This projects the tensor k1 onto the q direction
id k1.v1?{p1,p2,p3,p4} = k1.q * v1.q * prop(q.q);
.sort:tensor-reduction;

* Replace k1.q 
id k1.q = 1/2*Den0 - 1/2*Den1 + 1/2*q.q;
.sort:scalarized;

repeat;
    id Den0 * (1/Den0) = 1;
    id Den1 * (1/Den1) = 1;
endrepeat;
id (1/Den0) * (1/Den1) = B0(q.q, Melec^2, Melec^2);
id 1/Den0 = A0(Melec^2);
id 1/Den1 = A0(Melec^2);
* Clean up any remaining terms
id A0(0) = 0;
id Den0 = 0;
id Den1 = 0;
id B0(q.q,?a) = B0(t,?a);
.sort:final-clean;

id prop(q.q) = 1/t;
id q = p1-p3;
id Melec = 0;
id Mmuon = 0;
#call Mandelstam2To2(p1,p2,p3,p4,0,0,0,0)
id u = -s -t;
.sort:kinematics;

Local Ratio = MInt/MsqLO;
id t^(-1) = t/t^2;
id t^(-2) = 1/t^2;
id t^(-3) = 1/t^3;
.sort

Bracket A0, B0;
Format;
Print MsqLO ;
Print MInt;
Print Ratio;
* Save to file 
Format C;
#write <RenormRatio.txt> "%e;", Ratio;
.end

FORM 5.0.0 (Jan 27 2026, v5.0.0)                 Run: Sat Apr 18 01:13:39 2026
    
    * Process: Renormalization
    
    #-

   MsqLO =
       + 2*pow(e,4) + 4*s*pow(t,-1)*pow(e,4) + 4*pow(s,2)*pow(t,-2)*pow(e,4);

   MInt =
       + A0(pow(Melec,2)) * (  - 32*s*pow(t,-2)*pow(e,6) - 32*pow(s,2)*pow(
         t,-3)*pow(e,6) );

      MInt +=  + B0(t,pow(Melec,2),pow(Melec,2)) * ( 4*pow(e,6) + 16*s*pow(
         t,-1)*pow(e,6) + 16*pow(s,2)*pow(t,-2)*pow(e,6) );

      MInt +=  + 8*pow(t,-2)*pow(e,6);

   Ratio =
       + A0(pow(Melec,2)) * (  - 32/(2*pow(e,4) + 4*s*pow(t,-1)*pow(e,4) + 4*
         pow(s,2)*pow(t,-2)*pow(e,4))*s*pow(t,-2)*pow(e,6) - 32/(2*pow(e,4) + 
         4*s*pow(t,-1)*pow(e,4) + 4*pow(s,2)*pow(t,-2)*pow(e,4))*pow(s,2)*pow(
         t,-3)*pow(e,6) );

      Ratio +=  + B0(t,pow(Melec,2),pow(Melec,2)) * ( 4/(2*pow(e,4) + 4*s*pow(
         t,-1)*pow(e,4) + 4*pow(s,2)*pow(t,-2)*pow(e,4))*pow(e,6) + 16/(2*pow(
         e,4) + 4*s*pow(t,-1)*pow(e,4) + 4*pow(s,2)*pow(t,

In [126]:
import numpy as np
import sympy as sp

def B0_high_energy(t_val, m2, mu2, epsilon):
    return 1/epsilon - sp.log(-t_val/mu2) + 2

form_expr_Ratio = capture_physics_expr("scripts/RenormRatio.txt")
s, t, e, mu, eps = sp.symbols('s t e mu eps', real=True)
Melec = sp.symbols('Melec', real=True, positive=True)

# 2. Map the FORM-style strings to your SymPy symbols
# This ensures "B0(t, Melec^2, Melec^2)" becomes a symbolic function call
mapping = {
    's': s,
    't': t,
    'e': e,
    'mu': mu,
    'eps': eps,
    'Melec': Melec,
    'A0': sp.Function('A0'),
    'B0': sp.Function('B0')
}
ratio_expr = sp.parse_expr(form_expr_Ratio, local_dict=mapping)

ratio_limit = ratio_expr.subs({
    sp.Function('A0')(Melec**2): 0,
    sp.Function('B0')(t, Melec**2, Melec**2): B0_high_energy(t, Melec**2, mu**2, eps)
})

final_ratio = sp.simplify(ratio_limit)
print(final_ratio)
sp.pprint(final_ratio)


2*e**2*(2*eps + (eps*(2 - log(-t/mu**2)) + 1)*(4*s**2 + 4*s*t + t**2))/(eps*(2*s**2 + 2*s*t + t**2))
   2 ⎛        ⎛    ⎛       ⎛-t ⎞⎞    ⎞ ⎛   2            2⎞⎞
2⋅e ⋅⎜2⋅eps + ⎜eps⋅⎜2 - log⎜───⎟⎟ + 1⎟⋅⎝4⋅s  + 4⋅s⋅t + t ⎠⎟
     ⎜        ⎜    ⎜       ⎜ 2 ⎟⎟    ⎟                    ⎟
     ⎝        ⎝    ⎝       ⎝μ  ⎠⎠    ⎠                    ⎠
───────────────────────────────────────────────────────────
                      ⎛   2            2⎞                  
                  eps⋅⎝2⋅s  + 2⋅s⋅t + t ⎠                  
